<a href="https://colab.research.google.com/github/blufury/Class-projects/blob/master/Deep_Learning_Midterm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install motmetrics lap ultralytics supervision opencv-python-headless

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.4/217.4 kB 15.4 MB/s eta 0:00:00


In [ ]:
import os
import cv2
import math
import json
import time
import shutil
import random
import zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

from google.colab import drive
from IPython.display import Video, display

In [ ]:
# ---------------------------
# USER SETTINGS
# ---------------------------
DRIVE_MOUNT = True
PROJECT_ROOT = "/content/video_tracking_project"
DATA_ROOT = os.path.join(PROJECT_ROOT, "data")
MOT16_ROOT = os.path.join(DATA_ROOT, "MOT16")
OUTPUT_ROOT = os.path.join(PROJECT_ROOT, "outputs")
MODEL_ROOT = os.path.join(PROJECT_ROOT, "models")

# Change this to one MOT16 sequence folder you want to test on
TEST_SEQUENCE = "MOT16-02"

# Detection / training
NUM_CLASSES = 2   # background + pedestrian
BATCH_SIZE = 2
NUM_EPOCHS = 3
LEARNING_RATE = 0.005
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005
NUM_WORKERS = 2

# Inference / tracking
DETECTION_SCORE_THRESHOLD = 0.70
IOU_MATCH_THRESHOLD = 0.30
MAX_MISSED_FRAMES = 20

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(PROJECT_ROOT, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(MODEL_ROOT, exist_ok=True)

print("DEVICE:", DEVICE)

In [ ]:
if DRIVE_MOUNT:
    drive.mount('/content/drive')

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def compute_iou(boxA, boxB):
    """
    box format: [x1, y1, x2, y2]
    """
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter_w = max(0, xB - xA)
    inter_h = max(0, yB - yA)
    inter = inter_w * inter_h

    areaA = max(0, boxA[2] - boxA[0]) * max(0, boxA[3] - boxA[1])
    areaB = max(0, boxB[2] - boxB[0]) * max(0, boxB[3] - boxB[1])
    union = areaA + areaB - inter + 1e-6

    return inter / union

def xywh_to_xyxy(x, y, w, h):
    return [x, y, x + w, y + h]

def draw_box(img, box, text="", color=(0, 255, 0), thickness=2):
    x1, y1, x2, y2 = [int(v) for v in box]
    cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness)
    if text:
        cv2.putText(
            img,
            text,
            (x1, max(20, y1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            color,
            2,
            cv2.LINE_AA
        )
    return img

In [ ]:
def parse_gt_file(gt_file_path):
    """
    MOT16 gt format:
    frame, id, bb_left, bb_top, bb_width, bb_height, conf, class, visibility
    """
    df = pd.read_csv(gt_file_path, header=None)
    df.columns = [
        "frame", "track_id", "bb_left", "bb_top", "bb_width", "bb_height",
        "conf", "class_id", "visibility"
    ]
    return df

def build_frame_annotations(gt_df, only_pedestrians=True, require_conf=True):
    """
    Returns dict:
    frame_idx -> {
        "boxes": [[x1,y1,x2,y2], ...],
        "labels": [1,1,...],
        "track_ids": [...]
    }
    """
    frame_dict = defaultdict(lambda: {"boxes": [], "labels": [], "track_ids": []})

    for _, row in gt_df.iterrows():
        if require_conf and int(row["conf"]) != 1:
            continue

        # In MOT16, class 1 is pedestrian
        if only_pedestrians and int(row["class_id"]) != 1:
            continue

        x1 = float(row["bb_left"])
        y1 = float(row["bb_top"])
        w = float(row["bb_width"])
        h = float(row["bb_height"])
        x2 = x1 + w
        y2 = y1 + h

        frame_idx = int(row["frame"])
        frame_dict[frame_idx]["boxes"].append([x1, y1, x2, y2])
        frame_dict[frame_idx]["labels"].append(1)  # pedestrian
        frame_dict[frame_idx]["track_ids"].append(int(row["track_id"]))

    return frame_dict

In [ ]:
class MOT16DetectionDataset(Dataset):
    def __init__(self, seq_dirs, transforms=None):
        self.transforms = transforms
        self.samples = []

        for seq_dir in seq_dirs:
            img_dir = os.path.join(seq_dir, "img1")
            gt_file = os.path.join(seq_dir, "gt", "gt.txt")

            if not os.path.exists(img_dir) or not os.path.exists(gt_file):
                continue

            gt_df = parse_gt_file(gt_file)
            annotations = build_frame_annotations(gt_df)

            image_files = sorted([
                f for f in os.listdir(img_dir)
                if f.lower().endswith((".jpg", ".jpeg", ".png"))
            ])

            for img_name in image_files:
                frame_idx = int(Path(img_name).stem)
                img_path = os.path.join(img_dir, img_name)

                frame_ann = annotations.get(frame_idx, {"boxes": [], "labels": [], "track_ids": []})
                self.samples.append({
                    "img_path": img_path,
                    "boxes": frame_ann["boxes"],
                    "labels": frame_ann["labels"],
                    "track_ids": frame_ann["track_ids"]
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img_path = sample["img_path"]

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = TF.to_tensor(img)

        boxes = torch.as_tensor(sample["boxes"], dtype=torch.float32)
        labels = torch.as_tensor(sample["labels"], dtype=torch.int64)
        track_ids = torch.as_tensor(sample["track_ids"], dtype=torch.int64)

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            track_ids = torch.zeros((0,), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]) if len(boxes) > 0 else torch.tensor([]),
            "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64),
            "track_ids": track_ids
        }

        if self.transforms is not None:
            img, target = self.transforms(img, target)

        return img, target

def collate_fn(batch):
    return tuple(zip(*batch))

In [ ]:
class SimpleDetectionTransform:
    def __init__(self, train=True):
        self.train = train

    def __call__(self, image, target):
        if self.train:
            if random.random() < 0.5:
                _, h, w = image.shape
                image = torch.flip(image, dims=[2])

                boxes = target["boxes"].clone()
                if len(boxes) > 0:
                    x1 = boxes[:, 0].clone()
                    x2 = boxes[:, 2].clone()
                    boxes[:, 0] = w - x2
                    boxes[:, 2] = w - x1
                    target["boxes"] = boxes

            if random.random() < 0.3:
                image = torch.clamp(image * random.uniform(0.85, 1.15), 0.0, 1.0)

        return image, target

In [ ]:
# Point this at your extracted MOT16 train folders
mot16_train_dir = os.path.join(MOT16_ROOT, "train")

seq_dirs = []
if os.path.exists(mot16_train_dir):
    for seq_name in sorted(os.listdir(mot16_train_dir)):
        full_seq = os.path.join(mot16_train_dir, seq_name)
        if os.path.isdir(full_seq):
            seq_dirs.append(full_seq)

print("Found training sequences:", len(seq_dirs))
for s in seq_dirs:
    print(" -", os.path.basename(s))

train_dataset = MOT16DetectionDataset(
    seq_dirs=seq_dirs,
    transforms=SimpleDetectionTransform(train=True)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn
)

print("Training samples:", len(train_dataset))

In [ ]:
def get_model(num_classes=2):
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")

    # Freeze backbone
    for param in model.backbone.parameters():
        param.requires_grad = False

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    return model

model = get_model(NUM_CLASSES).to(DEVICE)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(
    params,
    lr=LEARNING_RATE,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY
)

lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.1)

print("Model ready.")

In [ ]:
def train_one_epoch(model, optimizer, data_loader, device, epoch):
    model.train()
    epoch_loss = 0.0

    for step, (images, targets) in enumerate(data_loader):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) if torch.is_tensor(v) else v for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        loss_value = losses.item()
        epoch_loss += loss_value

        if step % 20 == 0:
            print(f"Epoch {epoch} | Step {step}/{len(data_loader)} | Loss: {loss_value:.4f}")

    return epoch_loss / max(1, len(data_loader))

In [ ]:
history = []

for epoch in range(NUM_EPOCHS):
    avg_loss = train_one_epoch(model, optimizer, train_loader, DEVICE, epoch + 1)
    lr_scheduler.step()
    history.append(avg_loss)
    print(f"Epoch {epoch + 1} complete. Avg loss: {avg_loss:.4f}")

detector_ckpt = os.path.join(MODEL_ROOT, "fasterrcnn_pedestrian_mot16.pth")
torch.save(model.state_dict(), detector_ckpt)
print("Saved detector to:", detector_ckpt)

In [ ]:
inference_model = get_model(NUM_CLASSES).to(DEVICE)
inference_model.load_state_dict(torch.load(detector_ckpt, map_location=DEVICE))
inference_model.eval()

print("Inference model loaded.")

In [ ]:
@torch.no_grad()
def detect_people(model, image_bgr, score_threshold=0.7, device=DEVICE):
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    tensor = TF.to_tensor(image_rgb).to(device)

    outputs = model([tensor])[0]

    boxes = outputs["boxes"].detach().cpu().numpy()
    scores = outputs["scores"].detach().cpu().numpy()
    labels = outputs["labels"].detach().cpu().numpy()

    detections = []
    for box, score, label in zip(boxes, scores, labels):
        if label == 1 and score >= score_threshold:
            detections.append({
                "box": box.tolist(),
                "score": float(score),
                "label": int(label)
            })
    return detections

In [ ]:
class SingleTargetTracker:
    def __init__(self, iou_threshold=0.3, max_missed=20):
        self.iou_threshold = iou_threshold
        self.max_missed = max_missed
        self.active = False
        self.track_id = 1
        self.current_box = None
        self.missed = 0

    def initialize(self, detections):
        if len(detections) == 0:
            return None

        # Pick the largest detection as initial target
        best_det = None
        best_area = -1

        for det in detections:
            x1, y1, x2, y2 = det["box"]
            area = max(0, x2 - x1) * max(0, y2 - y1)
            if area > best_area:
                best_area = area
                best_det = det

        if best_det is not None:
            self.active = True
            self.current_box = best_det["box"]
            self.missed = 0

        return best_det

    def update(self, detections):
        if not self.active:
            return self.initialize(detections)

        best_det = None
        best_iou = -1

        for det in detections:
            iou = compute_iou(self.current_box, det["box"])
            if iou > best_iou:
                best_iou = iou
                best_det = det

        if best_det is not None and best_iou >= self.iou_threshold:
            self.current_box = best_det["box"]
            self.missed = 0
            return best_det

        self.missed += 1
        if self.missed > self.max_missed:
            self.active = False
            self.current_box = None

        return None

In [ ]:
def run_tracking_on_sequence(sequence_dir, output_video_path, model):
    img_dir = os.path.join(sequence_dir, "img1")
    image_files = sorted([
        f for f in os.listdir(img_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

    if len(image_files) == 0:
        raise ValueError(f"No frames found in {img_dir}")

    first_frame = cv2.imread(os.path.join(img_dir, image_files[0]))
    h, w = first_frame.shape[:2]

    fps = 15
    seqinfo_path = os.path.join(sequence_dir, "seqinfo.ini")
    if os.path.exists(seqinfo_path):
        with open(seqinfo_path, "r") as f:
            for line in f:
                if line.startswith("frameRate"):
                    fps = int(line.strip().split("=")[1])

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_video_path, fourcc, fps, (w, h))

    tracker = SingleTargetTracker(
        iou_threshold=IOU_MATCH_THRESHOLD,
        max_missed=MAX_MISSED_FRAMES
    )

    tracking_log = []

    for idx, img_name in enumerate(image_files):
        frame_path = os.path.join(img_dir, img_name)
        frame = cv2.imread(frame_path)

        detections = detect_people(
            model,
            frame,
            score_threshold=DETECTION_SCORE_THRESHOLD,
            device=DEVICE
        )

        if idx == 0:
            tracked = tracker.initialize(detections)
        else:
            tracked = tracker.update(detections)

        vis = frame.copy()

        # Optional: draw all detections in blue
        for det in detections:
            vis = draw_box(vis, det["box"], text=f"{det['score']:.2f}", color=(255, 0, 0), thickness=1)

        # Draw tracked target in green
        if tracked is not None and tracker.current_box is not None:
            vis = draw_box(vis, tracker.current_box, text=f"Target ID {tracker.track_id}", color=(0, 255, 0), thickness=3)
            tracking_log.append({
                "frame": idx + 1,
                "track_id": tracker.track_id,
                "box": [float(x) for x in tracker.current_box]
            })
        else:
            tracking_log.append({
                "frame": idx + 1,
                "track_id": None,
                "box": None
            })

        writer.write(vis)

        if idx % 50 == 0:
            print(f"Processed frame {idx + 1}/{len(image_files)}")

    writer.release()

    log_path = output_video_path.replace(".mp4", "_tracking_log.json")
    with open(log_path, "w") as f:
        json.dump(tracking_log, f, indent=2)

    return output_video_path, log_path

In [ ]:
test_sequence_dir = os.path.join(MOT16_ROOT, "test", TEST_SEQUENCE)
output_video_path = os.path.join(OUTPUT_ROOT, f"{TEST_SEQUENCE}_tracked.mp4")

video_path, log_path = run_tracking_on_sequence(
    sequence_dir=test_sequence_dir,
    output_video_path=output_video_path,
    model=inference_model
)

print("Saved video to:", video_path)
print("Saved tracking log to:", log_path)

In [ ]:
display(Video(video_path, embed=True))

In [ ]:
SAVE_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/video_tracking_project_outputs"

if SAVE_TO_DRIVE:
    ensure_dir(DRIVE_OUTPUT_DIR)
    shutil.copy(video_path, os.path.join(DRIVE_OUTPUT_DIR, os.path.basename(video_path)))
    shutil.copy(log_path, os.path.join(DRIVE_OUTPUT_DIR, os.path.basename(log_path)))
    shutil.copy(detector_ckpt, os.path.join(DRIVE_OUTPUT_DIR, os.path.basename(detector_ckpt)))
    print("Copied outputs to:", DRIVE_OUTPUT_DIR)